<a href="https://colab.research.google.com/github/sreejamanthena10/travel_agent/blob/main/Untitled6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys

# 1. Clean up the problematic versions
!pip uninstall -y pyOpenSSL cryptography deprecated langchain

# 2. Install stable, basic versions
!pip install -q pyOpenSSL==24.0.0 cryptography==42.0.0 deprecated==1.2.14 langchain langchain-google-genai duckduckgo-search requests streamlit

# 3. Setup the API Key
from getpass import getpass
if 'GOOGLE_API_KEY' not in os.environ:
    os.environ['GOOGLE_API_KEY'] = getpass('Enter your Google API Key: ')

# 4. Install the tunnel tool
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

print('\n✅ Environment fixed. Please restart your session (Runtime > Restart Session) and then run this cell once more.')

Found existing installation: pyOpenSSL 24.0.0
Uninstalling pyOpenSSL-24.0.0:
  Successfully uninstalled pyOpenSSL-24.0.0
Found existing installation: cryptography 42.0.0
Uninstalling cryptography-42.0.0:
  Successfully uninstalled cryptography-42.0.0
Found existing installation: Deprecated 1.2.14
Uninstalling Deprecated-1.2.14:
  Successfully uninstalled Deprecated-1.2.14
Found existing installation: langchain 1.3.1
Uninstalling langchain-1.3.1:
  Successfully uninstalled langchain-1.3.1
Enter your Google API Key: ··········
(Reading database ... 118216 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.5.0) over (2026.5.0) ...
Setting up cloudflared (2026.5.0) ...
Processing triggers for man-db (2.10.2-1) ...

✅ Environment fixed. Please restart your session (Runtime > Restart Session) and then run this cell once more.


In [2]:
print('Installing langchain-community...')
!pip install -q langchain-community

Installing langchain-community...


In [3]:
!pip install -U duckduckgo-search ddgs

In [4]:
%%writefile tools.py
import requests
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

# Tool 1: Web Search (No API key needed)
web_search = DuckDuckGoSearchRun()

# Tool 2: Custom API with simple error handling
@tool
def get_weather(location: str) -> str:
    """Use this tool to find the current weather for a specific city or destination."""
    try:
        # We use wttr.in, a free text-based weather API
        response = requests.get(f"https://wttr.in/{location}?format=3", timeout=5)
        response.raise_for_status() # Checks for internet errors
        return f"Weather in {location}: {response.text}"
    except Exception as error:
        return f"Error getting weather data: {error}"

# Group the tools together
my_tools = [web_search, get_weather]

Overwriting tools.py


In [5]:
%%writefile agent.py
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from tools import my_tools
import json

class SimpleAgentExecutor:
    def __init__(self, llm, tools, verbose=True):
        self.llm = llm.bind_tools(tools)
        self.tools = {t.name: t for t in tools}
        self.verbose = verbose

    def invoke(self, inputs):
        query = inputs["input"]
        messages = [HumanMessage(content=query)]
        res = self.llm.invoke(messages)

        if res.tool_calls:
            messages.append(res)
            for tool_call in res.tool_calls:
                name = tool_call["name"]
                args = tool_call["args"]
                if self.verbose: print(f"\n[Calling Tool: {name} with {args}]")

                observation = self.tools[name].invoke(args)
                messages.append(ToolMessage(content=str(observation), tool_call_id=tool_call["id"]))

            final_res = self.llm.invoke(messages)
            return {"output": final_res.content}

        return {"output": res.content}

def get_agent():
    # Using gemini-flash-latest based on available model list
    llm = ChatGoogleGenerativeAI(model="gemini-flash-latest")
    return SimpleAgentExecutor(llm=llm, tools=my_tools, verbose=True)

Overwriting agent.py


In [6]:
import sys
import importlib
import langchain

# Force a clean reload of agent and tools to apply the model selection
for mod in ['agent', 'tools']:
    if mod in sys.modules:
        del sys.modules[mod]

try:
    from agent import get_agent

    print("Initializing agent with Gemini Flash (Latest)...")
    executor = get_agent()

    query = "What is the weather in London?"
    print(f"\nQuerying: {query}")
    response = executor.invoke({"input": query})
    print(f"\nResult: {response['output']}")

except Exception as e:
    print(f"Error: {e}")

Initializing agent with Gemini Flash (Latest)...

Querying: What is the weather in London?
Error: Error calling model 'gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 4.854050459s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tie

In [7]:
import google.generativeai as genai
from google.colab import userdata
import os

# Configure the API key if not already set in environment
api_key = os.environ.get('GOOGLE_API_KEY')
if api_key:
    genai.configure(api_key=api_key)

print("Listing available models:")
try:
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            print(f"- {m.name}")
except Exception as e:
    print(f"Error listing models: {e}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Listing available models:
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-4-26b-a4b-it
- models/gemma-4-31b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-3-pro-preview
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3.1-flash-lite
- models/gemini-3-pro-image-preview
- models/nano-banana-pro-preview
- models/gemini-3.1-flash-image-preview
- models/gemini-3.5-flash
- models/lyria-3-clip-preview
- models/lyria-3-pro-preview
- models/gemini-3.1-flash-tts-preview
- models/gemini-robotics-er-1.5-preview
- models/gemini-robotics-er-1.6-preview
- m

In [8]:
!pip show langchain
import os
import langchain
langchain_path = os.path.dirname(langchain.__file__)
print(f"\nContents of {langchain_path}/agents:")
!ls {langchain_path}/agents

Name: langchain
Version: 1.3.1
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 

Contents of /usr/local/lib/python3.12/dist-packages/langchain/agents:
factory.py  __init__.py  middleware  __pycache__  structured_output.py


In [9]:
import langchain
import os
print(f"Location: {langchain.__file__}")
print(f"Version: {langchain.__version__}")
# If this still says 1.3.1, the restart didn't clear the cache correctly.

Location: /usr/local/lib/python3.12/dist-packages/langchain/__init__.py
Version: 1.3.1


In [10]:
%%writefile app.py
import streamlit as st
from agent import get_agent

st.title("My Multi-Tool Agent 🛠️")
st.write("I can search the web and check the weather for you!")

# Start the agent once
if "agent" not in st.session_state:
    st.session_state.agent = get_agent()

# Create a place to store chat history
if "messages" not in st.session_state:
    st.session_state.messages = []

# Show past messages
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# User input box
user_input = st.chat_input("Ask me a question...")

if user_input:
    # 1. Show what the user typed
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.write(user_input)

    # 2. Get the AI's answer using tools
    with st.chat_message("assistant"):
        with st.spinner("Using tools to find the answer..."):
            result = st.session_state.agent.invoke({"input": user_input})
            answer = result["output"]
            st.write(answer)
            st.session_state.messages.append({"role": "assistant", "content": answer})

Overwriting app.py


In [11]:
import os
import time
import subprocess

# 1. Clean up existing processes
print('Cleaning up old processes...')
os.system('pkill -f streamlit')
os.system('pkill -f cloudflared')
time.sleep(2)

# 2. Start Streamlit
print('Starting Streamlit...')
with open('streamlit.log', 'w') as f:
    subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501'], stdout=f, stderr=f)

# 3. Start Cloudflare Tunnel
print('Starting Cloudflare Tunnel...')
with open('tunnel.log', 'w') as f:
    subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8501'], stdout=f, stderr=f)

# 4. Wait and extract the link
print('Waiting for the public URL (this may take up to 15 seconds)...')
time.sleep(15)

print('\n--- Deployment Status ---')
link = !grep -o 'https://[-0-9a-z]*\.trycloudflare.com' tunnel.log | head -n 1
if link:
    print(f'✅ Streamlit is live at: {link[0]}')
else:
    print('❌ Link not found yet. Check the raw logs below for errors:')
    !cat tunnel.log
print('-------------------------')

Cleaning up old processes...
Starting Streamlit...
Starting Cloudflare Tunnel...
Waiting for the public URL (this may take up to 15 seconds)...

--- Deployment Status ---
✅ Streamlit is live at: https://sandra-booth-touring-cartridges.trycloudflare.com
-------------------------
